# Finish the medium training run

Ready for **760 remaining games (95 batches)** from the preserved model after the Herald repair. Learning from all 1,240 updated medium-run games is retained. One additional game in the interrupted batch had no update and will be replaced. This completes the original 2,000-game medium learning budget.

**Click Run → Run All Cells when ready.** Nothing has been launched during preparation. The first step validates the checkpoint, then measures a new baseline before training. Evaluations follow at 296 and 760 additional games. Original runs and saved models remain unchanged.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import HTML, display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'tools/train_neural.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the complete death-knight-lab checkout.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from hearthstone_sim.neural.dashboard import Dashboard

RUN_MODE = "medium"  # preflight, trial, medium, full
START_TRAINING = True  # Run All is an explicit request to launch the long run.
TOTAL_TRAINING_GAMES = 760  # Finish the original medium budget: 1,240 updated + 760 remaining.
EVALUATION_AT = [296, 760]  # Original medium milestones 1,536 and 2,000.
GAMES_PER_ITERATION = 8
MAX_ACTIONS = 1000
SEED = 71
DEVICE = 'cpu'
THREADS = 2  # PyTorch threads only; games remain sequential.
FAILURE_FRACTION = 0.02
FAILURE_MIN_ATTEMPTS = 100
FAILURE_REPEATS = 5
RETRY_CAP = 20
TRACE_FIRST = 6
TRACE_EVERY = 16
assert RUN_MODE in ("preflight", "trial", "medium", "full")
EVALUATE_AT_START = True  # Resumed weights at local game 0, before any updates.
CHECKPOINT_EVERY_ITERATIONS = 16  # 128 games; smaller intervals use more disk.
EVALUATION_EVERY_ITERATIONS = 64  # Roughly every 512 training games.
EVALUATION_GAMES_PER_OPPONENT = 64
EVALUATION_SEED = 1_000_000_000
RESUME_CHECKPOINT = ROOT / 'runs/neural_migrations/herald-repaired-medium-finish/training-000206-herald-migrated.pt'
ALLOW_CODE_UPGRADE = False  # Strict resume; the offline Herald migration is complete.
assert TOTAL_TRAINING_GAMES > 0 and TOTAL_TRAINING_GAMES % GAMES_PER_ITERATION == 0
ITERATIONS = TOTAL_TRAINING_GAMES // GAMES_PER_ITERATION
command = [sys.executable, str(ROOT / 'tools/train_neural.py'),
    '--iterations', str(ITERATIONS), '--games-per-iteration', str(GAMES_PER_ITERATION),
    '--max-actions', str(MAX_ACTIONS), '--seed', str(SEED), '--device', DEVICE,
    '--threads', str(THREADS), '--checkpoint-every-iterations', str(CHECKPOINT_EVERY_ITERATIONS),
    '--evaluation-every-iterations', str(EVALUATION_EVERY_ITERATIONS),
    '--evaluation-games-per-opponent', str(EVALUATION_GAMES_PER_OPPONENT),
    '--evaluation-seed', str(EVALUATION_SEED)]
if RESUME_CHECKPOINT is not None:
    command += ['--resume', str(RESUME_CHECKPOINT)]
if ALLOW_CODE_UPGRADE:
    command += ['--allow-code-upgrade']
command += ['--failure-fraction', str(FAILURE_FRACTION), '--failure-min-attempts', str(FAILURE_MIN_ATTEMPTS), '--failure-repeats', str(FAILURE_REPEATS), '--retry-cap', str(RETRY_CAP), '--trace-first', str(TRACE_FIRST), '--trace-every', str(TRACE_EVERY)]
command += ['--evaluate-at-start' if EVALUATE_AT_START else '--no-evaluate-at-start']
if RUN_MODE == 'trial':
    command += ['--evaluation-at', '200', '400']
elif RUN_MODE == 'medium':
    # Continuation milestones, plus a repaired-engine baseline at 0.
    command += ['--evaluation-at', *map(str, EVALUATION_AT)]
print(f'Ready to check: {TOTAL_TRAINING_GAMES:,} additional training games; {ITERATIONS:,} PPO batches.')
print('No training or evaluation has started in this cell.')



In [ ]:
# Read-only validation: no games, no evaluation and no weight updates.
preflight = subprocess.run(command + ['--preflight'], cwd=ROOT, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(preflight.stdout)
PREFLIGHT_OK = preflight.returncode == 0
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight failed. The training cell will refuse to launch.')


In [ ]:
if not START_TRAINING or RUN_MODE == "preflight":
    print('Preflight only. Set RUN_MODE to trial or full and START_TRAINING to True to launch.')
elif not globals().get('PREFLIGHT_OK', False):
    raise RuntimeError('Run the setup and successful preflight cells before starting.')
else:
    dashboard = Dashboard(TOTAL_TRAINING_GAMES)
    screen = display(HTML(dashboard.html()), display_id=True)
    LAST_CHECKPOINT = None
    RUN_FOLDER = None
    succeeded = False
    with subprocess.Popen(command + ['--train'], cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                try:
                    event = json.loads(line)
                except ValueError:
                    print(line.rstrip())
                    continue
                dashboard.consume(event)
                screen.update(HTML(dashboard.html()))
                if event['event'] == 'started':
                    RUN_FOLDER = event['folder']
                    print('Run folder:', RUN_FOLDER)
                elif event['event'] == 'iteration_saved' and event.get('checkpoint'):
                    LAST_CHECKPOINT = event['checkpoint']['path']
                    print('Resume checkpoint:', LAST_CHECKPOINT)
                elif event['event'] == 'evaluation_finished':
                    print('Detailed matchup evaluation:', event['report'])
                elif event['event'] == 'complete':
                    succeeded = True
                    LAST_CHECKPOINT = event['checkpoint']['path']
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            dashboard.status = 'Interrupted — resume from the latest durable checkpoint'
            screen.update(HTML(dashboard.html()))
            raise
    if returncode or not succeeded:
        dashboard.status = 'Stopped with an error — see output above; earlier checkpoints are preserved'
        screen.update(HTML(dashboard.html()))
        raise RuntimeError('Run did not finish. Do not delete the run folder; use the latest printed checkpoint.')
    print('Completed. Set RESUME_CHECKPOINT to this path for your next launch:', LAST_CHECKPOINT)


## Preserved state and safety
The continuation uses the explicit offline migration in `tools/migrate_herald_resume.py`. Weights, Adam, RNGs, next unused seed, opponents and safety totals are preserved. The unupdated partial batch is replaced, not counted as a completed learning update. Only the repaired Herald signature’s repeat counter is retired; other limits remain enabled. Older evaluations are archived separately because the engine changed. Keep every run folder and checkpoint receipt. See `docs/HERALD-RESUME.md` for the routing evidence and continuation details.
